In [1]:
import yaml
import camelot
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# pd.set_option('display.max_rows', 500)
# pd.set_option('display.max_columns', 500)
# pd.set_option('display.width', 1000)

C:\Users\XY\AppData\Local\Temp\ipykernel_27672\1552179835.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
with open('config.yaml', 'rb') as yaml_file:
    config=yaml.safe_load(yaml_file)

In [3]:
embeddings = HuggingFaceEmbeddings(model_name='google/embeddinggemma-300m')

In [4]:
pdf_filepath = 'fy2024_analysis_of_revenue_and_expenditure.pdf'
docs = PyPDFLoader(pdf_filepath).load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['chunk_size'], 
    chunk_overlap=config['chunk_overlap'],
    separators=['\n']
)

#TODO - switch to table detection model/method for scaling
table_pages = [8, 11, 12, 16, 19, 20, 24, 25, 26, 27, 
               28, 29, 30, 31, 32, 33, 34]

chunks = splitter.split_documents(docs)

non_table_chunks = [chunk for chunk in chunks 
                    if int(chunk.metadata['page_label']) not in table_pages]

In [5]:
# check = [chunk for chunk in chunks if int(chunk.metadata['page_label']) in table_pages]

# for chunk in check:
#     for line in chunk.page_content.split('\n'):
#         if 'table' in line.lower():
#             print(line)
#             print('')

In [6]:
row_chunk_list = []
for page in table_pages:
    page_chunks = [chunk for chunk in chunks if int(chunk.metadata['page_label']) == page]
    page_metadata = page_chunks[0].metadata

    table_name = ''
    for chunk in page_chunks:
        for line in chunk.page_content.split('\n'):
            if 'table' in line.lower():
                table_name = ' '.join(line.split())
    # print(table_name)

    tables = camelot.read_pdf(pdf_filepath, pages=str(page), flavor='ml')
    if len(tables) > 0:
        for table in tables:
            # table.df = table.df.replace('\n', '')
            table.df = table.df.replace({'\n': ''}, regex=True)
            table.df = table.df.replace({'BLANK': ''}, regex=True)
            table.df = table.df.replace({'  ': ' '}, regex=True)
            table.df = table.df[~table.df.eq('').all(axis=1)]

            # document specific
            if table.df[0].values[0].strip() == '' and table.df[0].values[1].strip() == '' :
                table.df = pd.concat([
                    table.df.iloc[:2].agg(' '.join).to_frame().T,
                    table.df.iloc[2:]
                ], ignore_index=True)
            table.df.columns = table.df.iloc[0]
            table.df = table.df.drop(0).reset_index(drop=True)
            col_list = table.df.columns.tolist()
            col_list[0] = '__item'
            col_list = [' '.join(col.split()) for col in col_list]
            table.df.columns = col_list
            
            # display(table.df)
            # print(table.df.columns)
            # row_chunk_list = []
            for index, row in table.df.iterrows():
                row_number = int(str(index))
                row_chunk_str = f'In the {table_name} within page {page}, row {row_number}\n'
                
                item_name = ''
                for index, col in enumerate(col_list):
                    if col == '__item':
                        item_name = row.iloc[index]
                    else:
                        if row.iloc[index] not in ['-', '']:
                            # TODO - flag negative value?
                            row_chunk_str += f'{item_name}, {col} was {row.iloc[index]}\n'
                # row_chunk_list.append(row_chunk_str)
                # print(row_chunk_str)
            
            # row_chunk = '\n'.join(row_chunk_list)
            # print(row_chunk)
            # print('-----------------------------------------')
                page_metadata['table_name'] = table_name
                page_metadata['row_number'] = row_number
                # page_metadata['table_df_json'] = table.df.to_json(orient='index')
                row_chunk_doc = Document(
                    page_content=row_chunk_str,
                    metadata=page_metadata
                )
                row_chunk_list.append(row_chunk_doc)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.
c:\zdrive\miniconda3\envs\htx_case\Lib\site-packages\torch\nn\modules\module.py:2588: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
c:\zdrive\miniconda3\envs\htx_case\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.weight: copying from a non-meta par

In [10]:
all_chunks_list = non_table_chunks + row_chunk_list

In [13]:
len(all_chunks_list)

420

In [12]:
non_table_chunks[10]

Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2024-02-26T11:32:07+08:00', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_actionid': 'a9460921-7f46-4542-8ed3-cc5668e11605', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_contentbits': '0', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_enabled': 'true', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_method': 'Privileged', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_name': 'Sensitive Normal', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_setdate': '2024-01-29T15:00:53Z', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_siteid': '0b11c524-9a1c-4e1b-84cb-6336aefc2243', 'moddate': '2024-02-26T11:32:09+08:00', 'source': 'fy2024_analysis_of_revenue_and_expenditure.pdf', 'total_pages': 37, 'page': 5, 'page_label': '6'}, page_content='Casino Taxes.  \n \nGoods and Services Tax collections are revised to $16.4 billion, which is             \n$1.0 billion ( 5.8%) lower than the Estimated FY2023 figure

In [9]:
row_chunk_list[10]

Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2024-02-26T11:32:07+08:00', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_actionid': 'a9460921-7f46-4542-8ed3-cc5668e11605', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_contentbits': '0', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_enabled': 'true', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_method': 'Privileged', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_name': 'Sensitive Normal', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_setdate': '2024-01-29T15:00:53Z', 'msip_label_153db910-0838-4c35-bb3a-1ee21aa199ac_siteid': '0b11c524-9a1c-4e1b-84cb-6336aefc2243', 'moddate': '2024-02-26T11:32:09+08:00', 'source': 'fy2024_analysis_of_revenue_and_expenditure.pdf', 'total_pages': 37, 'page': 7, 'page_label': '8', 'table_name': 'Table 1.1 Fiscal Position in FY2022 and FY2023', 'row_number': 10}, page_content='In the Table 1.1 Fiscal Position in FY2022 and FY2023 within page 8, row 10\nBetting 

In [ ]:
# for chunk in non_table_chunks:
    # print(chunk.metadata['page_label'])

In [ ]:
# for page in range(len(docs)):
#     tables = camelot.read_pdf(pdf_filepath, pages=str(page), flavor='ml')
#     if len(tables) > 0:
#         print(f'Page {page}: {len(tables)} tables')
#         for table in tables:
#             # display(table.df)
#             print(table.shape)

In [ ]:
# tables = camelot.read_pdf(pdf_filepath, pages='34', flavor='ml')
# print(len(tables))
# for index, table in enumerate(tables):
#     print(index)
#     display(table.df)

In [ ]:
# chunk_list = []
# for file in pdf_files:
#     # file = 'D&D 5E Starter Set Rulebook.pdf'
#     file_path = os.path.join(config['source_dir'], file)

#     loader = PyPDFLoader(file_path)
#     docs = loader.load()
#     splitter = RecursiveCharacterTextSplitter(
#         chunk_size=config['chunk_size'], 
#         chunk_overlap=config['chunk_overlap'],
#         separators=['\n']
#     )

#     chunks = splitter.split_documents(docs)
#     for chunk in chunks:
#         chunk.metadata['source'] = file

#     chunk_list += chunks